# Synapse submission — 8 remaining activation functions (Instance Norm / basic nnU-Net)

Adapted from `Frozon_alpha_2_Synapse.ipynb`. Same pipeline, same predictor settings, same
uint8/zip packaging — but loops over **8 activation cells** instead of one, and reads the trained
models from **`temp_nnunet_results`**.

Activations: `ReLU, ELU, GELU, Mish, ELiSH, HardELiSH, Logish, Smish`

**Output:** one Synapse zip per activation → 8 zips, each scored independently on the 219-case
blinded validation set by the same lesion-wise pipeline as the reported cells.

### Design notes

- **Model folders are discovered, not assumed.** Cell 2 scans `temp_nnunet_results` and matches
  folders to activations by *exact token*, not substring. Substring matching is unsafe here:
  `"ELU"` is contained in `"GELU"` and `"LeakyReLU"`, `"ReLU"` in `"LeakyReLU"`, `"ELiSH"` in
  `"HardELiSH"`. Token matching splits `nnUNetTrainer_Abl_GELU__nnUNetPlans__3d_fullres` on `__`
  then `_` and compares whole tokens, so each activation resolves to exactly one folder.
- **Resumable.** Inference and zipping both skip any activation already complete. Safe to
  re-run after a disconnect — 8 × 5-fold ensembles over 219 cases is many hours.
- **No checkpoint cleaning.** Same as the source notebook. Instance Norm has no running
  statistics, so nothing needs stripping either way.
- **Fold audit per activation.** Requires `checkpoint_final.pth`; falls back to
  `checkpoint_latest.pth` with a loud warning and records it in the manifest, because a cell
  ensembled from unfinished folds is not comparable to a complete one.

## 0 — Install, then RESTART the runtime  ⚠️

**Run this cell first, on its own. It will restart the session. That is expected.**

`pip install nnunetv2==2.2.1` pulls in NumPy 2.5.x, which breaks two ways:

- The already-imported NumPy in the live kernel is stale relative to the newly installed
  Python-level code, giving
  `AttributeError: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'`
  on `import nnunetv2...`. Only a kernel restart clears that.
- `numba` (a transitive nnU-Net dependency) requires `numpy<2.1`, so 2.5.x is genuinely
  incompatible, not merely stale.

So NumPy is pinned to `<2.1` and the runtime is restarted. **After the restart, continue from
cell 1 — do not re-run this cell** (it is idempotent and will simply report "already satisfied",
but re-running wastes a minute).

In [ ]:
import importlib.util, sys

def _numpy_ok():
    try:
        import numpy as _np
    except Exception:
        return False
    major, minor = (int(x) for x in _np.__version__.split(".")[:2])
    return (major, minor) < (2, 1)

already = importlib.util.find_spec("nnunetv2") is not None and _numpy_ok()

if already:
    import numpy as _np
    print(f"Already set up — nnunetv2 present, numpy {_np.__version__} (<2.1). No restart needed.")
else:
    print("Installing nnU-Net with numpy pinned <2.1 (numba requires it)...")
    !pip install -q "numpy<2.1" nnunetv2==2.2.1 nibabel
    print("\nInstall done. Restarting the runtime so the pinned NumPy is loaded cleanly.")
    print("After it restarts, run cell 1 next — skip this cell.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

## 1 — Mount & configure

`RESULTS_ROOT` is where the 8 trained models live. The cell tries the likely locations for
`temp_nnunet_results` and reports which it found; override manually if none match.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)

WORKSPACE = "/content/drive/MyDrive/nnU-Net Project/workspace"
DATASET   = "Dataset100_BraTS2023"

# --- the 8 remaining activations (screen at Instance Norm / basic nnU-Net) ---
ACTIVATIONS = ["ReLU", "ELU", "GELU", "Mish", "ELiSH", "HardELiSH", "Logish", "Smish"]

# --- where the trained models are: temp_nnunet_results ---
# Tried in order; first existing one wins. Override RESULTS_ROOT by hand if needed.
_candidates = [
    "/content/drive/MyDrive/temp_nnunet_results",
    os.path.join(WORKSPACE, "temp_nnunet_results"),
    "/content/drive/MyDrive/nnU-Net Project/temp_nnunet_results",
]
RESULTS_ROOT = next((p for p in _candidates if os.path.isdir(p)), None)

print("Drive mounted.")
for p in _candidates:
    print(f"   {'[FOUND]' if os.path.isdir(p) else '[     ]'} {p}")
if RESULTS_ROOT is None:
    print("\n   None of the candidates exist. Listing MyDrive top level to help you locate it:")
    for d in sorted(os.listdir("/content/drive/MyDrive"))[:60]:
        print("     -", d)
    raise FileNotFoundError("Set RESULTS_ROOT manually to the temp_nnunet_results folder.")
print(f"\n   RESULTS_ROOT = {RESULTS_ROOT}")

# --- nnU-Net environment ---
os.environ["nnUNet_raw"]          = os.path.join(WORKSPACE, "nnUNet_raw")
os.environ["nnUNet_preprocessed"] = os.path.join(WORKSPACE, "nnUNet_preprocessed")
os.environ["nnUNet_results"]      = RESULTS_ROOT          # <-- points at temp_nnunet_results

RAW_VAL_DIR      = os.path.join(WORKSPACE, "ASNR-MICCAI-BraTS2023-GLI-Challenge-ValidationData")
TARGET_IMAGES_TS = os.path.join(os.environ["nnUNet_raw"], DATASET, "imagesTs")
PRED_ROOT        = os.path.join(WORKSPACE, "Ensemble_Predictions")
ZIP_ROOT         = os.path.join(WORKSPACE, "Synapse_Submissions_IN8")
os.makedirs(TARGET_IMAGES_TS, exist_ok=True)
os.makedirs(PRED_ROOT, exist_ok=True)
os.makedirs(ZIP_ROOT, exist_ok=True)

print(f"   raw val:  {RAW_VAL_DIR}")
print(f"   preds ->  {PRED_ROOT}")
print(f"   zips  ->  {ZIP_ROOT}")
print(f"   plan:     {len(ACTIVATIONS)} activations {ACTIVATIONS}")

## 2 — Setup, trainer injection, and model discovery

Installs nnU-Net, applies the PyTorch `weights_only` patch, copies any custom trainer files from
Drive into the nnU-Net package, then **discovers** which results folder belongs to each activation.

Trainer files live in `workspace/trainer files/`:

- `custom_brats_activations_remaining.py` — the custom activation classes
- `nnUNetTrainer_IN_remaining.py` — the trainer classes for the 8 cells

The activations file is copied **first**, because the trainer file imports from it. The nnU-Net
trainer directory is also added to `sys.path`, so a plain `import custom_brats_activations_remaining`
resolves as well as a package-qualified one — nnU-Net imports trainers as members of
`nnunetv2.training.nnUNetTrainer`, under which a bare module-level import would otherwise fail.

The cell then reads the trainer file, lists the trainer classes it defines, and confirms each is
importable via nnU-Net's own class resolver — so an import error surfaces here rather than after
you've queued hours of inference.

In [ ]:
import os, shutil, glob, sys
import numpy as np

_maj, _min = (int(x) for x in np.__version__.split(".")[:2])
if (_maj, _min) >= (2, 1):
    raise RuntimeError(
        f"numpy {np.__version__} is too new (numba needs <2.1). Run cell 0 and let it restart."
    )
print(f"numpy {np.__version__} OK")

from tqdm import tqdm
import torch
import nnunetv2
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
print(f"nnunetv2 {getattr(nnunetv2, '__version__', '2.2.1')} imported OK")

# --- PyTorch weights_only patch (idempotent) ---
if not getattr(torch.load, "_weights_only_patch", False):
    _orig_load = torch.load
    def _patched_load(*args, **kwargs):
        kwargs["weights_only"] = False
        return _orig_load(*args, **kwargs)
    _patched_load._weights_only_patch = True
    torch.load = _patched_load
    print("PyTorch torch.load patched (weights_only=False).")
else:
    print("torch.load already patched.")

nnunet_trainer_dir = os.path.join(os.path.dirname(nnunetv2.__file__), "training", "nnUNetTrainer")

# --- Inject the custom trainer files -------------------------------------------------
# Order matters: the activations module is a dependency of the trainer module.
TRAINER_SRC_DIR = os.path.join(WORKSPACE, "trainer files")
TRAINER_FILES = [
    "custom_brats_activations_remaining.py",   # activation classes  (copy FIRST)
    "nnUNetTrainer_IN_remaining.py",           # trainer classes     (imports the above)
]

if not os.path.isdir(TRAINER_SRC_DIR):
    print(f"Trainer source dir not found: {TRAINER_SRC_DIR}")
    print("Contents of WORKSPACE:")
    for d in sorted(os.listdir(WORKSPACE)):
        print("   -", d)
    raise FileNotFoundError(TRAINER_SRC_DIR)

missing = [f for f in TRAINER_FILES if not os.path.exists(os.path.join(TRAINER_SRC_DIR, f))]
if missing:
    print(f"Missing from {TRAINER_SRC_DIR}: {missing}")
    print("Files actually present:")
    for d in sorted(os.listdir(TRAINER_SRC_DIR)):
        print("   -", d)
    raise FileNotFoundError(f"Missing trainer file(s): {missing}")

for fname in TRAINER_FILES:
    shutil.copy(os.path.join(TRAINER_SRC_DIR, fname),
                os.path.join(nnunet_trainer_dir, fname))
    print(f"   injected {fname}")

# Make bare `import custom_brats_activations_remaining` resolvable too: nnU-Net imports
# trainers as members of nnunetv2.training.nnUNetTrainer, so a module-level plain import
# inside the trainer file needs its directory on sys.path.
if nnunet_trainer_dir not in sys.path:
    sys.path.insert(0, nnunet_trainer_dir)
    print(f"   sys.path += {nnunet_trainer_dir}")

# --- List the trainer classes the file defines, and verify each is importable ---------
import re
from nnunetv2.utilities.find_class_by_name import recursive_find_python_class

with open(os.path.join(nnunet_trainer_dir, "nnUNetTrainer_IN_remaining.py")) as fh:
    trainer_src = fh.read()
TRAINER_CLASSES = re.findall(r"^class\s+(\w+)", trainer_src, flags=re.M)

print(f"\n{len(TRAINER_CLASSES)} class(es) defined in nnUNetTrainer_IN_remaining.py:")
importable, broken = [], []
for cls_name in TRAINER_CLASSES:
    try:
        found = recursive_find_python_class(nnunet_trainer_dir, cls_name,
                                           "nnunetv2.training.nnUNetTrainer")
    except Exception as e:
        found, err = None, e
    if found is not None:
        importable.append(cls_name); print(f"   OK       {cls_name}")
    else:
        broken.append(cls_name);     print(f"   FAILED   {cls_name}")

if not importable:
    raise RuntimeError(
        "No trainer class could be imported. Most likely the activations module import "
        "failed — check the import statement at the top of nnUNetTrainer_IN_remaining.py."
    )
if broken:
    print(f"\n   WARNING: {len(broken)} class(es) not importable: {broken}")
    print("   Base/helper classes failing here is harmless; a *cell* trainer failing is not.")

# --- Discover the results folder for each activation, by EXACT TOKEN match ---
# Substring matching is unsafe: 'ELU' occurs inside 'GELU'/'LeakyReLU', 'ELiSH' inside 'HardELiSH',
# 'ReLU' inside 'LeakyReLU'. Split 'X__nnUNetPlans__3d_fullres' on '__' then '_' and compare tokens.
def _tokens(folder_name):
    return [t.lower() for t in folder_name.split("__")[0].split("_")]

dataset_results = os.path.join(RESULTS_ROOT, DATASET)
if not os.path.isdir(dataset_results):
    print(f"\n{DATASET} not found directly under RESULTS_ROOT. Contents of RESULTS_ROOT:")
    for d in sorted(os.listdir(RESULTS_ROOT)):
        print("   -", d)
    raise FileNotFoundError(f"Expected {dataset_results}")

all_folders = sorted(d for d in os.listdir(dataset_results)
                     if os.path.isdir(os.path.join(dataset_results, d)))
print(f"\n{len(all_folders)} model folder(s) in {DATASET}:")
for d in all_folders:
    print("   -", d)

MODEL_DIRS, unresolved, ambiguous = {}, [], {}
for act in ACTIVATIONS:
    hits = [d for d in all_folders if act.lower() in _tokens(d)]
    if len(hits) == 1:
        MODEL_DIRS[act] = os.path.join(dataset_results, hits[0])
    elif len(hits) == 0:
        unresolved.append(act)
    else:
        ambiguous[act] = hits

print("\nResolved activation -> model folder:")
for act in ACTIVATIONS:
    if act in MODEL_DIRS:
        print(f"   {act:10s} -> {os.path.basename(MODEL_DIRS[act])}")
    elif act in ambiguous:
        print(f"   {act:10s} -> AMBIGUOUS: {ambiguous[act]}")
    else:
        print(f"   {act:10s} -> NOT FOUND")

if ambiguous:
    raise RuntimeError(f"Ambiguous matches {ambiguous}. Resolve by editing MODEL_DIRS by hand.")
if unresolved:
    print(f"\nWARNING: no folder found for {unresolved}.")
    print("   These will be SKIPPED. If they are trained under a different name, add them manually:")
    print("   MODEL_DIRS['GELU'] = os.path.join(dataset_results, '<exact folder name>')")

# --- Cross-check: does each resolved activation have an importable trainer class? ------
# The folder name encodes the trainer that produced it; nnU-Net must be able to import that
# class to load the checkpoints. Mismatch here = inference failure, so catch it now.
print("\nCross-check (results folder <-> importable trainer class):")
no_class = []
for act in sorted(MODEL_DIRS):
    folder_trainer = os.path.basename(MODEL_DIRS[act]).split("__")[0]
    ok = folder_trainer in importable
    print(f"   {act:10s} {folder_trainer:42s} {'OK' if ok else 'NO IMPORTABLE CLASS'}")
    if not ok:
        no_class.append((act, folder_trainer))

if no_class:
    print("\n   WARNING — these will fail at inference:")
    for act, tr in no_class:
        print(f"      {act}: folder expects trainer '{tr}', not importable")
    print(f"   Classes that ARE importable: {importable}")
    print("   Either the trainer file does not define these, or the folder names differ from")
    print("   the class names. Fix before running inference.")

print(f"\n{len(MODEL_DIRS)}/{len(ACTIVATIONS)} activations resolved, "
      f"{len(MODEL_DIRS) - len(no_class)} with an importable trainer.")

## 2b — Fold audit (run before committing GPU hours)

Per activation: which folds exist, and whether each has `checkpoint_final.pth` (finished) or only
`checkpoint_latest.pth` (unfinished). A cell ensembled from unfinished folds is **not comparable**
to a complete 5-fold cell — the audit surfaces that before you spend hours on inference and submit
non-comparable numbers to Synapse.

In [ ]:
FOLD_INFO = {}
print(f"{'activation':11s} {'folds':>20s}  {'final':>5s} {'latest':>6s} {'missing':>7s}  status")
for act, mdir in MODEL_DIRS.items():
    final_f, latest_f, missing_f = [], [], []
    for fold in range(5):
        fd = os.path.join(mdir, f"fold_{fold}")
        if os.path.exists(os.path.join(fd, "checkpoint_final.pth")):
            final_f.append(fold)
        elif os.path.exists(os.path.join(fd, "checkpoint_latest.pth")):
            latest_f.append(fold)
        else:
            missing_f.append(fold)
    usable = sorted(final_f + latest_f)
    FOLD_INFO[act] = {"final": final_f, "latest": latest_f, "missing": missing_f, "usable": usable}
    if len(final_f) == 5:
        status = "COMPLETE"
    elif not usable:
        status = "NO CHECKPOINTS"
    elif latest_f:
        status = f"INCOMPLETE (folds {latest_f} unfinished)"
    else:
        status = f"PARTIAL ({len(usable)}/5 folds)"
    print(f"{act:11s} {str(usable):>20s}  {len(final_f):5d} {len(latest_f):6d} {len(missing_f):7d}  {status}")

ready    = [a for a, i in FOLD_INFO.items() if len(i["final"]) == 5]
degraded = [a for a, i in FOLD_INFO.items() if i["usable"] and len(i["final"]) < 5]
empty    = [a for a, i in FOLD_INFO.items() if not i["usable"]]

print(f"\n   fully complete (5/5 final): {ready}")
if degraded:
    print(f"   DEGRADED — will run but are not comparable: {degraded}")
if empty:
    print(f"   no checkpoints, will be skipped: {empty}")

# Set to True to run only the fully-complete cells (recommended for the reported table).
STRICT_5FOLD_ONLY = False
RUN_LIST = ready if STRICT_5FOLD_ONLY else ready + degraded
print(f"\n   RUN_LIST ({len(RUN_LIST)}): {RUN_LIST}")

## 3 — Stage the validation data (idempotent)

Copies the 219 raw validation cases into nnU-Net's `imagesTs` naming. Skips files already present,
so re-running is free. Channel map: `0000` T1 / `0001` T1c / `0002` T2 / `0003` FLAIR, accepting both
the legacy (`t1`, `t1ce`, `t2`, `flair`) and BraTS-2023 (`t1n`, `t1c`, `t2w`, `t2f`) suffixes.

In [ ]:
print("Staging validation data...")
if not os.path.isdir(RAW_VAL_DIR):
    raise FileNotFoundError(f"Raw validation folder not found: {RAW_VAL_DIR}")

patient_folders = [f.path for f in os.scandir(RAW_VAL_DIR) if f.is_dir()]
copied = 0
for pf in tqdm(patient_folders, desc="Copying & renaming"):
    pid = os.path.basename(pf)
    for fp in glob.glob(os.path.join(pf, "*.nii.gz")):
        fn = os.path.basename(fp).lower()
        cid = None
        if   fn.endswith("t1.nii.gz")  or fn.endswith("t1n.nii.gz"):   cid = "0000"
        elif fn.endswith("t1c.nii.gz") or fn.endswith("t1ce.nii.gz"):  cid = "0001"
        elif fn.endswith("t2.nii.gz")  or fn.endswith("t2w.nii.gz"):   cid = "0002"
        elif fn.endswith("flair.nii.gz") or fn.endswith("t2f.nii.gz"): cid = "0003"
        if cid:
            tgt = os.path.join(TARGET_IMAGES_TS, f"{pid}_{cid}.nii.gz")
            if not os.path.exists(tgt):
                shutil.copy(fp, tgt); copied += 1

n_files = len([f for f in os.listdir(TARGET_IMAGES_TS) if f.endswith(".nii.gz")])
NUM_CASES = n_files // 4
print(f"Staged {copied} new file(s). {n_files} files = {NUM_CASES} cases.")
if n_files % 4 != 0:
    print(f"   WARNING: {n_files} is not divisible by 4 — a modality may be missing for some case.")

## 4 — Inference: 5-fold soft-voting ensemble per activation

One pass per activation, resumable. An activation is skipped if its output folder already holds
`NUM_CASES` masks. Predictor settings are identical to the source notebook so the outputs are
directly comparable to the already-reported cells.

**This is the long step** — 8 activations × 5 folds × 219 cases. Re-run the cell after any
disconnect; completed activations are skipped.

In [ ]:
import re, inspect, importlib, sys
import torch.nn as nn
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from nnunetv2.utilities.find_class_by_name import recursive_find_python_class
from batchgenerators.utilities.file_and_folder_operations import load_json, join
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager

# Define TRAINER_DIR and TRAINER_PKG
TRAINER_DIR = os.path.join(os.path.dirname(nnunetv2.__file__), "training", "nnUNetTrainer")
TRAINER_PKG = "nnunetv2.training.nnUNetTrainer"

mod = importlib.import_module(f"{TRAINER_PKG}.custom_brats_activations_remaining")

# --- test: can the class-level call work as nnU-Net does it? ---
a0    = sorted(MODEL_DIRS)[0]
mdir  = MODEL_DIRS[a0]
plans = PlansManager(load_json(join(mdir, "plans.json")))
dj    = load_json(join(mdir, "dataset.json"))
cm    = plans.get_configuration("3d_fullres")
n_in  = len(dj.get("channel_names", dj.get("modality", {})))
tname = "nnUNetTrainer_500ep_" + a0
cls   = recursive_find_python_class(TRAINER_DIR, tname, TRAINER_PKG)

needs_patch = False
try:
    net = cls.build_network_architecture(plans, dj, cm, n_in, enable_deep_supervision=False)
    print("class-level call works — no patch needed")
    del net
except Exception as e:
    needs_patch = True
    print(f"class-level call FAILS: {type(e).__name__}: {e}")
    print("-> patching all trainers to static form\n")

if needs_patch:
    # unbind swap_activations so it can be used without an instance
    raw_swap = None
    for k in ("swap_activations",):
        for c in cls.__mro__:
            if k in c.__dict__:
                raw_swap = c.__dict__[k]; break
        if raw_swap: break
    params = list(inspect.signature(raw_swap).parameters)
    swap = (lambda net, act: raw_swap(None, net, act)) if params[0] == "self" else raw_swap

    def make_bna(act):
        def bna(plans_manager, dataset_json, configuration_manager,
                num_input_channels, enable_deep_supervision=True):
            net = nnUNetTrainer.build_network_architecture(
                plans_manager, dataset_json, configuration_manager,
                num_input_channels, enable_deep_supervision)
            return swap(net, act)
        return staticmethod(bna)

    patched = []
    for name in dir(mod):
        obj = getattr(mod, name)
        if not (inspect.isclass(obj) and name.startswith("nnUNetTrainer_500ep_")):
            continue
        try:
            src = inspect.getsource(obj)
        except Exception:
            continue
        m = re.search(r"swap_activations\s*\(\s*network\s*,\s*([A-Za-z_][\w\.]*)", src)
        if not m:
            print(f"  {name}: no swap_activations call found — SKIPPED"); continue
        act = eval(m.group(1), {"nn": nn, **vars(mod)})
        setattr(obj, "build_network_architecture", make_bna(act))
        patched.append((name, m.group(1)))
        print(f"  patched {name:32s} -> {m.group(1)}")

    print(f"\n{len(patched)} trainer(s) patched")

    # verify: build each and count the real activation modules
    print("\n" + "=" * 62)
    for act_name in sorted(MODEL_DIRS):
        t = "nnUNetTrainer_500ep_" + act_name
        c = recursive_find_python_class(TRAINER_DIR, t, TRAINER_PKG)
        if c is None:
            print(f"  {act_name:11s} unresolvable"); continue
        try:
            net = c.build_network_architecture(plans, dj, cm, n_in, enable_deep_supervision=False)
            kinds = {}
            for m_ in net.modules():
                n_ = type(m_).__name__
                if n_ in ("ReLU","LeakyReLU","ELU","GELU","SiLU","Mish","PReLU",
                          "ELiSH","HardELiSH","Logish","Smish"):
                    kinds[n_] = kinds.get(n_, 0) + 1
            ok = act_name in kinds or (act_name == "Swish" and "SiLU" in kinds)
            print(f"  {act_name:11s} {'OK  ' if ok else 'MISMATCH'} {kinds}")
            del net
        except Exception as e:
            print(f"  {act_name:11s} build failed: {type(e).__name__}: {e}")

In [ ]:
import os, time, traceback

def n_masks(folder):
    return len([f for f in os.listdir(folder) if f.endswith('.nii.gz')]) if os.path.isdir(folder) else 0

TODO = [a for a in sorted(MODEL_DIRS)]
print('activations to process:', TODO)
print('expected cases per activation:', NUM_CASES)

summary = []
for idx, act in enumerate(TODO, 1):
    mdir  = MODEL_DIRS[act]
    folds = FOLD_INFO[act]['usable']
    ckpt  = 'checkpoint_final.pth' if not FOLD_INFO[act]['latest'] else 'checkpoint_latest.pth'
    out   = os.path.join(PRED_ROOT, 'Validation_IN_' + act)
    os.makedirs(out, exist_ok=True)

    print()
    print('=' * 66)
    print('  [%d/%d]  %s   folds=%s   ckpt=%s' % (idx, len(TODO), act, tuple(folds), ckpt))
    print('=' * 66)

    done = n_masks(out)
    if done >= NUM_CASES:
        print('  already complete (%d/%d) - skipping.' % (done, NUM_CASES))
        summary.append({'activation': act, 'folds': folds, 'ckpt': ckpt,
                        'masks': done, 'status': 'skipped (complete)'})
        continue
    if done:
        print('  resuming: %d/%d masks already present' % (done, NUM_CASES))

    t0 = time.time()
    try:
        predictor = nnUNetPredictor(
            tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
            perform_everything_on_gpu=True,
            device=torch.device('cuda', 0),
            verbose=False, verbose_preprocessing=False, allow_tqdm=True,
        )
        predictor.initialize_from_trained_model_folder(
            mdir, use_folds=tuple(folds), checkpoint_name=ckpt)
        predictor.predict_from_files(
            TARGET_IMAGES_TS, out,
            save_probabilities=False,
            overwrite=False,
            num_processes_segmentation_export=1,
            folder_with_segs_from_prev_stage=None,
            num_parts=1, part_id=0,
        )
        del predictor; torch.cuda.empty_cache()
        got  = n_masks(out)
        mins = (time.time() - t0) / 60
        print('  done: %d/%d masks in %.0f min' % (got, NUM_CASES, mins))
        summary.append({'activation': act, 'folds': folds, 'ckpt': ckpt, 'masks': got,
                        'status': 'ok' if got == NUM_CASES else 'INCOMPLETE (%d/%d)' % (got, NUM_CASES),
                        'minutes': round(mins, 1)})
    except Exception as e:
        torch.cuda.empty_cache()
        print('  FAILED: %s: %s' % (type(e).__name__, e))
        traceback.print_exc(limit=5)
        summary.append({'activation': act, 'folds': folds, 'ckpt': ckpt,
                        'masks': n_masks(out), 'status': 'FAILED (%s)' % type(e).__name__})
        continue

print()
print('=' * 66)
print('INFERENCE SUMMARY')
print('=' * 66)
for s in summary:
    print('  %-11s %4d/%d  %-24s %s' % (s['activation'], s['masks'], NUM_CASES, s['ckpt'], s['status']))

In [ ]:
# ============================================================
# ReLU: define the missing trainer, verify, infer, zip
# ============================================================
import os, sys, time, inspect, importlib, traceback, zipfile
import numpy as np, nibabel as nib, torch
import nnunetv2
from nnunetv2.utilities.find_class_by_name import recursive_find_python_class
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager
from batchgenerators.utilities.file_and_folder_operations import load_json, join

TRAINER_DIR = os.path.join(nnunetv2.__path__[0], 'training', 'nnUNetTrainer')
TRAINER_PKG = 'nnunetv2.training.nnUNetTrainer'
HOST        = os.path.join(TRAINER_DIR, 'custom_brats_activations_remaining.py')

RELU_SRC = '''

# --- added for inference: ReLU cell (static form, no runtime patch needed) ---
class nnUNetTrainer_500ep_ReLU(BaseAblationTrainer):
    @staticmethod
    def build_network_architecture(plans_manager, dataset_json, configuration_manager,
                                   num_input_channels, enable_deep_supervision=True):
        import inspect as _insp
        from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer as _nnb
        _net = _nnb.build_network_architecture(plans_manager, dataset_json,
                                               configuration_manager, num_input_channels,
                                               enable_deep_supervision)
        _sw = None
        for _c in BaseAblationTrainer.__mro__:
            if "swap_activations" in _c.__dict__:
                _sw = _c.__dict__["swap_activations"]
                break
        if _sw is None:
            raise RuntimeError("swap_activations not found on BaseAblationTrainer")
        _p = list(_insp.signature(_sw).parameters)
        return _sw(None, _net, nn.ReLU) if _p and _p[0] == "self" else _sw(_net, nn.ReLU)
'''

with open(HOST) as fh:
    cur = fh.read()
if 'class nnUNetTrainer_500ep_ReLU' in cur:
    print('nnUNetTrainer_500ep_ReLU already in file')
else:
    with open(HOST, 'a') as fh:
        fh.write(RELU_SRC)
    print('appended nnUNetTrainer_500ep_ReLU ->', HOST)

for k in [k for k in sys.modules if 'custom_brats' in k or 'remaining' in k]:
    del sys.modules[k]
importlib.invalidate_caches()

cls = recursive_find_python_class(TRAINER_DIR, 'nnUNetTrainer_500ep_ReLU', TRAINER_PKG)
print('resolvable:', cls is not None)
if cls is None:
    raise RuntimeError('still unresolvable - check the append above')

# --- verify it really builds a ReLU network ---
mdir  = MODEL_DIRS['ReLU']
plans = PlansManager(load_json(join(mdir, 'plans.json')))
dj    = load_json(join(mdir, 'dataset.json'))
cm    = plans.get_configuration('3d_fullres')
n_in  = len(dj.get('channel_names', dj.get('modality', {})))
net   = cls.build_network_architecture(plans, dj, cm, n_in, enable_deep_supervision=False)
kinds = {}
for m in net.modules():
    n = type(m).__name__
    if n in ('ReLU','LeakyReLU','ELU','GELU','SiLU','Mish','PReLU','ELiSH','HardELiSH','Logish','Smish'):
        kinds[n] = kinds.get(n, 0) + 1
print('activation census:', kinds)
del net; torch.cuda.empty_cache()
assert set(kinds) == {'ReLU'}, 'expected ReLU only, got %s' % kinds

# --- inference ---
folds = FOLD_INFO['ReLU']['usable']
out   = os.path.join(PRED_ROOT, 'Validation_IN_ReLU')
os.makedirs(out, exist_ok=True)
have  = len([f for f in os.listdir(out) if f.endswith('.nii.gz')])
print('\nReLU: %d/%d masks present, folds=%s' % (have, NUM_CASES, tuple(folds)))

if have < NUM_CASES:
    t0 = time.time()
    predictor = nnUNetPredictor(
        tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
        perform_everything_on_gpu=True, device=torch.device('cuda', 0),
        verbose=False, verbose_preprocessing=False, allow_tqdm=True)
    predictor.initialize_from_trained_model_folder(
        mdir, use_folds=tuple(folds), checkpoint_name='checkpoint_final.pth')
    predictor.predict_from_files(
        TARGET_IMAGES_TS, out, save_probabilities=False, overwrite=False,
        num_processes_segmentation_export=1, folder_with_segs_from_prev_stage=None,
        num_parts=1, part_id=0)
    del predictor; torch.cuda.empty_cache()
    have = len([f for f in os.listdir(out) if f.endswith('.nii.gz')])
    print('done: %d/%d in %.0f min' % (have, NUM_CASES, (time.time()-t0)/60))

# --- zip ---
if have == NUM_CASES:
    zp = os.path.join(ZIP_ROOT, 'BraTS2023_Submission_IN_ReLU.zip')
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fn in [f for f in os.listdir(out) if f.endswith('.nii.gz')]:
            img  = nib.load(os.path.join(out, fn))
            data = img.get_fdata().astype(np.uint8)
            cln  = nib.Nifti1Image(data, img.affine, img.header)
            cln.set_data_dtype(np.uint8)
            name = fn.replace('ensemble_pred_', '')
            tmp  = os.path.join('/content', name)
            nib.save(cln, tmp); zf.write(tmp, arcname=name); os.remove(tmp)
    print('zipped -> %s  (%.0f MB)' % (zp, os.path.getsize(zp)/1e6))
else:
    print('NOT zipped - only %d/%d masks' % (have, NUM_CASES))

## 5 — Package for Synapse (one zip per activation)

Casts labels to `uint8` (Synapse requires integer labels) and zips with the bare case filenames —
no folder nesting, no filename prefix. Resumable: an existing zip is skipped unless
`OVERWRITE_ZIPS = True`.

In [ ]:
import zipfile
import numpy as np
import nibabel as nib

OVERWRITE_ZIPS = True
zip_report = []

for act in RUN_LIST:
    out = os.path.join(PRED_ROOT, f"Validation_IN_{act}")
    zp  = os.path.join(ZIP_ROOT, f"BraTS2023_Submission_IN_{act}.zip")

    if not os.path.isdir(out):
        print(f"  {act:11s} no prediction folder — skipping."); continue
    files = [f for f in os.listdir(out) if f.endswith(".nii.gz")]
    if not files:
        print(f"  {act:11s} no masks — skipping."); continue
    if os.path.exists(zp) and not OVERWRITE_ZIPS:
        mb = os.path.getsize(zp) / 1e6
        print(f"  {act:11s} zip exists ({mb:.0f} MB) — skipping. Set OVERWRITE_ZIPS=True to rebuild.")
        zip_report.append({"activation": act, "n": len(files), "zip": zp, "status": "existing"})
        continue

    if len(files) != NUM_CASES:
        print(f"  {act:11s} WARNING: {len(files)} masks, expected {NUM_CASES} — zipping anyway.")

    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fn in tqdm(files, desc=f"{act:11s}", leave=False):
            img = nib.load(os.path.join(out, fn))
            data = img.get_fdata().astype(np.uint8)
            clean = nib.Nifti1Image(data, img.affine, img.header)
            clean.set_data_dtype(np.uint8)
            name = fn.replace("ensemble_pred_", "")
            tmp = os.path.join("/content", name)
            nib.save(clean, tmp)
            zf.write(tmp, arcname=name)
            os.remove(tmp)
    mb = os.path.getsize(zp) / 1e6
    print(f"  {act:11s} {len(files)} masks -> {os.path.basename(zp)} ({mb:.0f} MB)")
    zip_report.append({"activation": act, "n": len(files), "zip": zp, "status": "built"})

print(f"\n{len(zip_report)} zip(s) in {ZIP_ROOT}")

## 6 — Manifest

Writes `submission_manifest_IN8.csv` alongside the zips: which activation, which folds, which
checkpoint, how many masks. Submit one zip at a time to Synapse and record the returned
lesion-wise scores against this manifest — the fold/checkpoint columns are what let you tell later
whether a cell was a complete 5-fold ensemble.

In [ ]:
import os, glob
import pandas as pd

def n_masks(folder):
    return len([f for f in os.listdir(folder) if f.endswith(".nii.gz")]) if os.path.isdir(folder) else 0

# in-memory results if the inference cell ran in this session; otherwise fall back to disk
mem = {s["activation"]: s for s in summary} if "summary" in dir() and summary else {}
zips = {os.path.basename(p).replace("BraTS2023_Submission_IN_", "").replace(".zip", ""): p
        for p in glob.glob(os.path.join(ZIP_ROOT, "BraTS2023_Submission_IN_*.zip"))}

acts = sorted(set(list(MODEL_DIRS) if "MODEL_DIRS" in dir() else [])
              | set(mem) | set(zips))
if not acts:
    acts = sorted(os.path.basename(p).replace("Validation_IN_", "")
                  for p in glob.glob(os.path.join(PRED_ROOT, "Validation_IN_*")))

expected = NUM_CASES if "NUM_CASES" in dir() else None
rows = []
for a in acts:
    pred = os.path.join(PRED_ROOT, f"Validation_IN_{a}")
    got  = n_masks(pred)
    fi   = FOLD_INFO.get(a, {}) if "FOLD_INFO" in dir() else {}
    used = fi.get("usable", [])
    ckpt = mem.get(a, {}).get("ckpt") or (
        "checkpoint_final.pth" if not fi.get("latest") else "checkpoint_latest.pth")
    zp   = zips.get(a, "")
    if got == 0:                      status = "not run"
    elif expected and got < expected: status = f"INCOMPLETE ({got}/{expected})"
    else:                             status = "ok"
    rows.append({
        "activation": a,
        "norm": "IN (basic nnU-Net)",
        "folds_used": ",".join(map(str, used)),
        "n_folds": len(used),
        "checkpoint": ckpt,
        "complete_5fold_final": len(fi.get("final", [])) == 5 if fi else None,
        "n_masks": got,
        "expected": expected,
        "status": status,
        "zip": os.path.basename(zp),
        "zip_MB": round(os.path.getsize(zp) / 1e6, 1) if zp else None,
    })

if not rows:
    print("Nothing to report — no predictions, no zips, no MODEL_DIRS.")
    print("Run the inference cell first.")
else:
    man = pd.DataFrame(rows).sort_values("activation")
    os.makedirs(ZIP_ROOT, exist_ok=True)
    man_path = os.path.join(ZIP_ROOT, "submission_manifest_IN8.csv")
    man.to_csv(man_path, index=False)
    print(man.to_string(index=False))
    print(f"\nmanifest -> {man_path}")

    ready = man[(man.status == "ok") & (man.zip != "")]
    print(f"\nready to submit: {len(ready)}/{len(man)}  {list(ready.activation)}")
    pend = man[man.status != "ok"]
    if len(pend):
        print("not ready:")
        for _, r in pend.iterrows():
            print(f"   {r.activation}: {r.status}")
    bad = man[man.complete_5fold_final == False]
    if len(bad):
        print("\nNOT comparable to a complete 5-fold cell — flag in any table:")
        for _, r in bad.iterrows():
            print(f"   {r.activation}: {r.n_folds} folds, {r.checkpoint}")

## 7 — After Synapse scoring

For each activation, download the per-case CSV from Synapse and keep the filename tied to the
activation (e.g. `lesionwise_IN_GELU.csv`). Those per-case files are what the selection rule needs —
fold-level means alone cannot support a patient-paired test.

**Before applying any selection rule:** it must already be written down. With the eight fold-mean
results seen so far, between-activation spread (SD ≈ 0.0027) is roughly a third of the within-
activation fold spread (SD ≈ 0.0073), so expect few or no activations to separate from the
LeakyReLU baseline after correction. That is a legitimate finding —
but the rule has to be fixed in advance for it to count.